# TFG V4 vs V5 comparison

This notebook compares the result files generated by `TFG_V4.ipynb` and `TFG_V5.ipynb` after the full parameter/mixer/method sweeps.

It validates that both notebooks define the same benchmark cases, then compares:

- best V4 vs best V5 configuration per use case;
- optimization method performance (`Experimental`, `M1`, `M2`, `M3`, `M4`, `M5 hybrid`);
- mixer methods (`local_geometric`, `linear_valid`, `rx_all`);
- V5 transition modes;
- full configuration rankings, including `(transition_mode, mixer_method, optimization method)`;
- selected optimal parameters (`theta`, `beta`, `r`) for the best rows.

The method CSVs (`*_param_methods_all_cases.csv`) are the source of truth because they contain every simulated row. The `*_all_config_*` CSVs generated by V4/V5 are loaded as consistency checks and direct summaries when present.


In [1]:
import csv
import json
import os
import re
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", os.path.join(os.getcwd(), ".matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", os.path.join(os.getcwd(), ".cache"))
os.environ.setdefault("MPLBACKEND", "Agg")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except Exception:
    pass

plt.rcParams.update({
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
})

V4_COLOR = "#3498db"
V5_COLOR = "#f39c12"
V5_ORIENTED_COLOR = "#9b59b6"
BASELINE_COLOR = "0.45"
POS_COLOR = "#2ecc71"
NEG_COLOR = "#e74c3c"

ROOT = Path(".")
if not (ROOT / "TFG_V4.ipynb").exists() and (ROOT / "TFG" / "TFG_V4.ipynb").exists():
    ROOT = ROOT / "TFG"

V4_NOTEBOOK = ROOT / "TFG_V4.ipynb"
V5_NOTEBOOK = ROOT / "TFG_V5.ipynb"
V4_DIR = ROOT / "analysis_v4_cases"
V5_DIR = ROOT / "analysis_v5_cases"
OUT_DIR = ROOT / "analysis_v4_v5_comparison"
OUT_DIR.mkdir(exist_ok=True)

V4_SUMMARY_CSV = V4_DIR / "v4_case_study_summary.csv"
V5_SUMMARY_CSV = V5_DIR / "v5_case_study_summary.csv"
V4_METHODS_CSV = V4_DIR / "v4_param_methods_all_cases.csv"
V5_METHODS_CSV = V5_DIR / "v5_param_methods_all_cases.csv"
V4_ALL_CONFIG_BEST_CSV = V4_DIR / "v4_all_config_best_by_case.csv"
V5_ALL_CONFIG_BEST_CSV = V5_DIR / "v5_all_config_best_by_case.csv"
V4_ALL_CONFIG_SUMMARY_CSV = V4_DIR / "v4_all_config_summary_by_mixer_method.csv"
V5_ALL_CONFIG_SUMMARY_CSV = V5_DIR / "v5_all_config_summary_by_mode_mixer_method.csv"

METHOD_ORDER = ["Experimental", "M1 (sweep)", "M2 (eigen)", "M3 (theory)", "M4 (grad)", "M5 (hybrid)"]
MIXER_ORDER = ["local_geometric", "linear_valid", "rx_all"]
V5_MODE_ORDER = ["delta_cost", "descent", "valid_boundary", "oriented_valid"]


def savefig(fig, stem):
    fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight")
    fig.savefig(OUT_DIR / f"{stem}.png", dpi=200, bbox_inches="tight")
    plt.close(fig)


def slug(text):
    return re.sub(r"[^a-zA-Z0-9_]+", "_", str(text)).strip("_").lower()


def ordered_unique(values, preferred_order=None):
    values = [v for v in values if pd.notna(v)]
    if preferred_order is None:
        return sorted(set(values))
    present = set(values)
    return [v for v in preferred_order if v in present] + sorted(present - set(preferred_order))


In [2]:
# =========================================================
# Validate that V4 and V5 use the same benchmark cases
# =========================================================

def extract_case_studies(notebook_path, variable_name, next_function_name):
    nb = json.loads(Path(notebook_path).read_text())
    source = "\n".join("".join(cell.get("source", [])) for cell in nb["cells"])
    start = source.find(f"{variable_name} = [")
    end = source.find(f"\n\ndef {next_function_name}", start)
    if start < 0 or end < 0:
        raise ValueError(f"Could not extract {variable_name} from {notebook_path}")
    block = source[start:end]
    ns = {"np": np}
    exec(block, ns)
    return ns[variable_name]


def normalize_case(case):
    return {
        "name": case["name"],
        "N": list(case["N"]),
        "M": list(case["M"]),
        "occupied_coords": [tuple(c) for c in case["occupied_coords"]],
        "theta": float(case["theta"]),
        "beta": float(case["beta"]),
        "mixer_method": case.get("mixer_method", "local_geometric"),
    }

v4_cases = [normalize_case(c) for c in extract_case_studies(V4_NOTEBOOK, "V4_CASE_STUDIES", "v4_slug")]
v5_cases = [normalize_case(c) for c in extract_case_studies(V5_NOTEBOOK, "V5_CASE_STUDIES", "v5_slug")]

if v4_cases != v5_cases:
    print("WARNING: V4_CASE_STUDIES and V5_CASE_STUDIES are not identical.")
    for a, b in zip(v4_cases, v5_cases):
        if a != b:
            print("Mismatch:", a["name"], a, b)
else:
    print(f"OK: V4 and V5 define the same {len(v4_cases)} use cases.")

case_order = [case["name"] for case in v4_cases]
case_labels = [name.replace("_", "\n", 2) for name in case_order]
case_meta = pd.DataFrame(v4_cases)
case_meta.to_csv(OUT_DIR / "v4_v5_case_definitions.csv", index=False)


OK: V4 and V5 define the same 10 use cases.


In [3]:
# =========================================================
# Load result CSV files and normalize column types
# =========================================================

def read_csv(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing result file: {path}")
    return pd.read_csv(path)


def optional_read_csv(path):
    return pd.read_csv(path) if path.exists() else None

v4_summary = read_csv(V4_SUMMARY_CSV)
v5_summary = read_csv(V5_SUMMARY_CSV)
v4_methods = read_csv(V4_METHODS_CSV)
v5_methods = read_csv(V5_METHODS_CSV)
v4_all_config_best_file = optional_read_csv(V4_ALL_CONFIG_BEST_CSV)
v5_all_config_best_file = optional_read_csv(V5_ALL_CONFIG_BEST_CSV)
v4_all_config_summary_file = optional_read_csv(V4_ALL_CONFIG_SUMMARY_CSV)
v5_all_config_summary_file = optional_read_csv(V5_ALL_CONFIG_SUMMARY_CSV)

# Older CSVs did not include every new categorical column. Add defaults so the
# comparison can still run and clearly reports what is missing.
if "mixer_method" not in v4_methods.columns:
    v4_methods["mixer_method"] = v4_methods["case"].map({c["name"]: c["mixer_method"] for c in v4_cases})
if "mixer_method" not in v5_methods.columns:
    v5_methods["mixer_method"] = v5_methods["case"].map({c["name"]: c["mixer_method"] for c in v5_cases})
if "transition_mode" not in v4_methods.columns:
    v4_methods["transition_mode"] = "cost_phase"
if "transition_mode" not in v4_summary.columns:
    v4_summary["transition_mode"] = "cost_phase"
if "mixer_method" not in v4_summary.columns:
    v4_summary["mixer_method"] = v4_summary["case"].map({c["name"]: c["mixer_method"] for c in v4_cases})
if "mixer_method" not in v5_summary.columns:
    v5_summary["mixer_method"] = v5_summary["case"].map({c["name"]: c["mixer_method"] for c in v5_cases})
for df in [v4_all_config_best_file, v4_all_config_summary_file]:
    if df is not None and "transition_mode" not in df.columns:
        df["transition_mode"] = "cost_phase"

numeric_cols = [
    "W", "valid_windows", "P_uniform", "theta_default", "beta_default",
    "r_star_default", "P_star_default", "theta_opt_heatmap", "beta_opt_heatmap",
    "P_opt_heatmap", "theta_opt_scan_default_beta", "P_opt_scan_default_beta",
    "theta", "theta_over_pi", "beta", "beta_over_pi", "r", "P_valid",
    "delta_theta_pi_vs_exp", "delta_beta_pi_vs_exp", "delta_r_vs_exp", "delta_P_vs_exp",
    "mean_P_valid", "median_P_valid", "std_P_valid", "min_P_valid", "max_P_valid",
    "wins_best_by_case", "cases",
]
all_loaded_frames = [
    v4_summary, v5_summary, v4_methods, v5_methods,
    v4_all_config_best_file, v5_all_config_best_file,
    v4_all_config_summary_file, v5_all_config_summary_file,
]
for df in [frame for frame in all_loaded_frames if frame is not None]:
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    if "case" in df.columns:
        df["case"] = pd.Categorical(df["case"], categories=case_order, ordered=True)

for name, df in [("V4 summary", v4_summary), ("V5 summary", v5_summary), ("V4 methods", v4_methods), ("V5 methods", v5_methods)]:
    missing = sorted(set(case_order) - set(df["case"].astype(str)))
    extra = sorted(set(df["case"].astype(str)) - set(case_order))
    print(f"{name}: rows={len(df)}, missing={missing}, extra={extra}")
    print(f"  mixers={ordered_unique(df.get('mixer_method', []), MIXER_ORDER)}")
    if "transition_mode" in df.columns:
        print(f"  transition_modes={ordered_unique(df['transition_mode'], ['cost_phase'] + V5_MODE_ORDER)}")
    if "method" in df.columns:
        print(f"  methods={ordered_unique(df['method'], METHOD_ORDER)}")

for label, df in [("V4 methods", v4_methods), ("V5 methods", v5_methods)]:
    mixers = ordered_unique(df["mixer_method"], MIXER_ORDER)
    if set(MIXER_ORDER) - set(mixers):
        print(f"WARNING: {label} does not contain all mixers. Re-run the last cell of the source notebook for complete mixer comparisons.")
    if "M5 (hybrid)" not in set(df["method"]):
        print(f"WARNING: {label} does not contain M5 (hybrid). Re-run the last cell of the source notebook after the M5 update.")

for label, df in [("V4 all-config best", v4_all_config_best_file), ("V5 all-config best", v5_all_config_best_file), ("V4 all-config summary", v4_all_config_summary_file), ("V5 all-config summary", v5_all_config_summary_file)]:
    if df is None:
        print(f"NOTE: {label} CSV is missing; this notebook will derive the same information from param_methods_all_cases.csv.")
    else:
        print(f"{label}: rows={len(df)}")


V4 summary: rows=10, missing=[], extra=[]
  mixers=['local_geometric', 'linear_valid']
  transition_modes=['cost_phase']
V5 summary: rows=40, missing=[], extra=[]
  mixers=['local_geometric', 'linear_valid']
  transition_modes=['delta_cost', 'descent', 'valid_boundary', 'oriented_valid']
V4 methods: rows=180, missing=[], extra=[]
  mixers=['local_geometric', 'linear_valid', 'rx_all']
  transition_modes=['cost_phase']
  methods=['Experimental', 'M1 (sweep)', 'M2 (eigen)', 'M3 (theory)', 'M4 (grad)', 'M5 (hybrid)']
V5 methods: rows=720, missing=[], extra=[]
  mixers=['local_geometric', 'linear_valid', 'rx_all']
  transition_modes=['delta_cost', 'descent', 'valid_boundary', 'oriented_valid']
  methods=['Experimental', 'M1 (sweep)', 'M2 (eigen)', 'M3 (theory)', 'M4 (grad)', 'M5 (hybrid)']
V4 all-config best: rows=10
V5 all-config best: rows=10
V4 all-config summary: rows=18
V5 all-config summary: rows=72


In [4]:
# =========================================================
# Helper functions and full-sweep comparison tables
# =========================================================

def best_rows(df, group_cols, score_col="P_valid"):
    df_sorted = df.sort_values(score_col, ascending=False)
    return df_sorted.drop_duplicates(group_cols).sort_values(group_cols).reset_index(drop=True)


def require_cases(df, label):
    missing = sorted(set(case_order) - set(df["case"].astype(str)))
    if missing:
        raise ValueError(f"{label} is missing cases: {missing}")
    return df


def add_version(df, version):
    out = df.copy()
    out["version"] = version
    return out


def normalize_best_file(df, version):
    if df is None:
        return None
    out = df.copy()
    if "transition_mode" not in out.columns:
        out["transition_mode"] = "cost_phase"
    out["source_version"] = version
    return out

# Main comparison: full sweep over every simulated categorical choice.
# The method CSVs are source of truth. The all-config best CSVs are checked
# against this derived result when present.
v4_best_case = best_rows(v4_methods, ["case"])
v5_best_case = best_rows(v5_methods, ["case"])
require_cases(v4_best_case, "V4 full-sweep best-by-case")
require_cases(v5_best_case, "V5 full-sweep best-by-case")
v4_best_case = v4_best_case.set_index("case").loc[case_order].reset_index()
v5_best_case = v5_best_case.set_index("case").loc[case_order].reset_index()

case_comparison = pd.DataFrame({
    "case": case_order,
    "P_uniform": v4_best_case["P_uniform"].to_numpy(),
    "V4_best_P_valid": v4_best_case["P_valid"].to_numpy(),
    "V4_best_method": v4_best_case["method"].to_numpy(),
    "V4_best_mixer": v4_best_case["mixer_method"].to_numpy(),
    "V4_best_transition_mode": v4_best_case["transition_mode"].to_numpy(),
    "V4_theta_over_pi": v4_best_case["theta_over_pi"].to_numpy(),
    "V4_beta_over_pi": v4_best_case["beta_over_pi"].to_numpy(),
    "V4_r": v4_best_case["r"].to_numpy(),
    "V5_best_P_valid": v5_best_case["P_valid"].to_numpy(),
    "V5_best_method": v5_best_case["method"].to_numpy(),
    "V5_best_mixer": v5_best_case["mixer_method"].to_numpy(),
    "V5_best_transition_mode": v5_best_case["transition_mode"].to_numpy(),
    "V5_theta_over_pi": v5_best_case["theta_over_pi"].to_numpy(),
    "V5_beta_over_pi": v5_best_case["beta_over_pi"].to_numpy(),
    "V5_r": v5_best_case["r"].to_numpy(),
})
case_comparison["delta_V5_minus_V4"] = case_comparison["V5_best_P_valid"] - case_comparison["V4_best_P_valid"]
case_comparison.to_csv(OUT_DIR / "v4_v5_best_by_case.csv", index=False)
print("Full-sweep comparison, optimized over all simulated combinations:")
print(case_comparison.to_string(index=False))

# Consistency checks with all_config_best_by_case generated inside V4/V5.
for label, file_df, derived_df in [("V4", v4_all_config_best_file, v4_best_case), ("V5", v5_all_config_best_file, v5_best_case)]:
    if file_df is None:
        continue
    file_best = file_df.set_index("case").loc[case_order].reset_index()
    max_diff = float(np.nanmax(np.abs(file_best["P_valid"].to_numpy() - derived_df["P_valid"].to_numpy())))
    if max_diff > 1e-10:
        print(f"WARNING: {label} all_config_best_by_case.csv differs from param_methods_all_cases-derived best rows (max |delta P|={max_diff:.3e}). Re-run the source notebook to refresh outputs.")
    else:
        print(f"OK: {label} all_config_best_by_case.csv matches param_methods_all_cases-derived best rows.")

# Secondary reference: default-mixer case-study heatmap overview from summary CSVs.
v4_default_case = v4_summary.sort_values("P_opt_heatmap", ascending=False).drop_duplicates("case")
v5_default_case = v5_summary.sort_values("P_opt_heatmap", ascending=False).drop_duplicates("case")
require_cases(v4_default_case, "V4 default-mixer case-study overview")
require_cases(v5_default_case, "V5 default-mixer case-study overview")
v4_default_case = v4_default_case.set_index("case").loc[case_order].reset_index()
v5_default_case = v5_default_case.set_index("case").loc[case_order].reset_index()

default_mixer_case_comparison = pd.DataFrame({
    "case": case_order,
    "P_uniform": v4_default_case["P_uniform"].to_numpy(),
    "V4_P_opt_heatmap": v4_default_case["P_opt_heatmap"].to_numpy(),
    "V4_mixer": v4_default_case["mixer_method"].to_numpy(),
    "V5_P_opt_heatmap": v5_default_case["P_opt_heatmap"].to_numpy(),
    "V5_mixer": v5_default_case["mixer_method"].to_numpy(),
    "V5_best_transition_mode": v5_default_case["transition_mode"].to_numpy(),
})
default_mixer_case_comparison["delta_V5_minus_V4"] = default_mixer_case_comparison["V5_P_opt_heatmap"] - default_mixer_case_comparison["V4_P_opt_heatmap"]
default_mixer_case_comparison.to_csv(OUT_DIR / "v4_v5_default_mixer_case_comparison.csv", index=False)
print("\nDefault-mixer case-study overview reference:")
print(default_mixer_case_comparison.to_string(index=False))


Full-sweep comparison, optimized over all simulated combinations:
                            case  P_uniform  V4_best_P_valid V4_best_method   V4_best_mixer V4_best_transition_mode  V4_theta_over_pi  V4_beta_over_pi  V4_r  V5_best_P_valid V5_best_method   V5_best_mixer V5_best_transition_mode  V5_theta_over_pi  V5_beta_over_pi  V5_r  delta_V5_minus_V4
           01_1d_tiny_single_gap   0.200000         0.760868    M5 (hybrid)    linear_valid              cost_phase          0.093750         0.062500    46         0.928341    M5 (hybrid)    linear_valid                 descent          0.250000         0.302083    30           0.167473
            02_1d_main_reference   0.285714         0.929910    M5 (hybrid)    linear_valid              cost_phase          0.093750         0.114583    33         0.937035    M5 (hybrid) local_geometric                 descent          0.093750         0.114583    18           0.007124
          03_1d_two_free_regions   0.375000         0.969108    M5 

In [5]:
# =========================================================
# Figure 1: full-sweep best probabilities by use case
# =========================================================

x = np.arange(len(case_comparison))
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - 0.24, case_comparison["P_uniform"], width=0.20, color="0.75", label="uniform K/W")
ax.bar(x, case_comparison["V4_best_P_valid"], width=0.24, color=V4_COLOR, label="V4 best full sweep")
ax.bar(x + 0.24, case_comparison["V5_best_P_valid"], width=0.24, color=V5_COLOR, label="V5 best full sweep")
for xi, row in case_comparison.iterrows():
    label = f"{row['V5_best_transition_mode']}\n{row['V5_best_mixer']}\n{row['V5_best_method']}"
    ax.text(xi + 0.24, min(1.06, row["V5_best_P_valid"] + 0.025), label, rotation=90, ha="center", va="bottom", fontsize=7)
ax.set_xticks(x)
ax.set_xticklabels(case_labels, rotation=45, ha="right")
ax.set_ylabel("P_valid")
ax.set_ylim(0, 1.14)
ax.set_title("V4 vs V5: full-sweep best P_valid by use case")
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.18))
fig.tight_layout()
savefig(fig, "v4_v5_case_probabilities")

fig, ax = plt.subplots(figsize=(12, 5))
delta = case_comparison["delta_V5_minus_V4"].to_numpy()
colors = [POS_COLOR if d >= 0 else NEG_COLOR for d in delta]
ax.bar(x, delta, color=colors, alpha=0.9)
ax.axhline(0, color=BASELINE_COLOR, linestyle="--", linewidth=1.2)
for xi, d, mode in zip(x, delta, case_comparison["V5_best_transition_mode"]):
    va = "bottom" if d >= 0 else "top"
    offset = 0.02 if d >= 0 else -0.02
    ax.text(xi, d + offset, f"{d:+.3f}\n{mode}", ha="center", va=va, fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(case_labels, rotation=45, ha="right")
ax.set_ylabel("Delta P_valid = V5 full sweep - V4 full sweep")
ax.set_title("V5 full-sweep advantage over V4")
fig.tight_layout()
savefig(fig, "v4_v5_delta_by_case")

# Reference figure for the narrower default-mixer case-study heatmap view.
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - 0.24, default_mixer_case_comparison["P_uniform"], width=0.20, color="0.75", label="uniform K/W")
ax.bar(x, default_mixer_case_comparison["V4_P_opt_heatmap"], width=0.24, color=V4_COLOR, label="V4 default-mixer heatmap")
ax.bar(x + 0.24, default_mixer_case_comparison["V5_P_opt_heatmap"], width=0.24, color=V5_COLOR, label="V5 default-mixer best mode")
for xi, row in default_mixer_case_comparison.iterrows():
    ax.text(xi + 0.24, min(1.06, row["V5_P_opt_heatmap"] + 0.025), row["V5_best_transition_mode"], rotation=90, ha="center", va="bottom", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(case_labels, rotation=45, ha="right")
ax.set_ylabel("P_valid")
ax.set_ylim(0, 1.14)
ax.set_title("Reference: default-mixer case-study heatmap comparison")
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.18))
fig.tight_layout()
savefig(fig, "v4_v5_default_mixer_case_probabilities")


In [6]:
# =========================================================
# Comparison by optimization method
# =========================================================

method_rows = []
method_case_rows = []
for method in ordered_unique(pd.concat([v4_methods["method"], v5_methods["method"]]), METHOD_ORDER):
    v4_m = v4_methods[v4_methods["method"] == method]
    v5_m = v5_methods[v5_methods["method"] == method]
    if v4_m.empty or v5_m.empty:
        continue
    v4_by_case = best_rows(v4_m, ["case"])
    v5_by_case = best_rows(v5_m, ["case"])
    merged = v4_by_case[["case", "P_valid", "mixer_method", "theta_over_pi", "beta_over_pi", "r"]].merge(
        v5_by_case[["case", "P_valid", "mixer_method", "transition_mode", "theta_over_pi", "beta_over_pi", "r"]],
        on="case", suffixes=("_V4", "_V5")
    )
    merged["method"] = method
    merged["delta_V5_minus_V4"] = merged["P_valid_V5"] - merged["P_valid_V4"]
    method_case_rows.append(merged)
    method_rows.append({
        "method": method,
        "V4_mean_P_valid": float(merged["P_valid_V4"].mean()),
        "V5_mean_P_valid": float(merged["P_valid_V5"].mean()),
        "mean_delta_V5_minus_V4": float(merged["delta_V5_minus_V4"].mean()),
        "V4_median_P_valid": float(merged["P_valid_V4"].median()),
        "V5_median_P_valid": float(merged["P_valid_V5"].median()),
        "V5_wins_cases": int((merged["delta_V5_minus_V4"] > 0).sum()),
        "cases": int(len(merged)),
    })

method_comparison = pd.DataFrame(method_rows)
method_detail = pd.concat(method_case_rows, ignore_index=True) if method_case_rows else pd.DataFrame()
method_comparison.to_csv(OUT_DIR / "v4_v5_method_comparison.csv", index=False)
method_detail.to_csv(OUT_DIR / "v4_v5_method_by_case_detail.csv", index=False)
print(method_comparison.to_string(index=False))

x_m = np.arange(len(method_comparison))
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x_m - 0.17, method_comparison["V4_mean_P_valid"], width=0.34, color=V4_COLOR, label="V4 mean")
ax.bar(x_m + 0.17, method_comparison["V5_mean_P_valid"], width=0.34, color=V5_COLOR, label="V5 mean")
ax.set_xticks(x_m)
ax.set_xticklabels(method_comparison["method"], rotation=20, ha="right")
ax.set_ylabel("mean best P_valid across use cases")
ax.set_ylim(0, 1.05)
ax.set_title("V4 vs V5 by optimization method")
ax.legend()
fig.tight_layout()
savefig(fig, "v4_v5_methods_mean")

fig, ax = plt.subplots(figsize=(10, 4))
colors = [POS_COLOR if d >= 0 else NEG_COLOR for d in method_comparison["mean_delta_V5_minus_V4"]]
ax.bar(x_m, method_comparison["mean_delta_V5_minus_V4"], color=colors)
ax.axhline(0, color=BASELINE_COLOR, linestyle="--", linewidth=1.2)
ax.set_xticks(x_m)
ax.set_xticklabels(method_comparison["method"], rotation=20, ha="right")
ax.set_ylabel("mean delta P_valid")
ax.set_title("Mean V5 - V4 gap by optimization method")
fig.tight_layout()
savefig(fig, "v4_v5_method_delta_mean")


      method  V4_mean_P_valid  V5_mean_P_valid  mean_delta_V5_minus_V4  V4_median_P_valid  V5_median_P_valid  V5_wins_cases  cases
Experimental         0.833473         0.867829                0.034356           0.841958           0.884755              7     10
  M1 (sweep)         0.888137         0.903576                0.015439           0.901100           0.919956              6     10
  M2 (eigen)         0.653433         0.698433                0.045000           0.659866           0.768742              6     10
 M3 (theory)         0.551005         0.514176               -0.036829           0.567237           0.482630              3     10
   M4 (grad)         0.432306         0.585970                0.153664           0.415424           0.572949              8     10
 M5 (hybrid)         0.907035         0.922533                0.015498           0.925754           0.932688              6     10


In [7]:
# =========================================================
# Mixer-method comparison
# =========================================================

mixer_rows = []
mixer_case_rows = []
for mixer in MIXER_ORDER:
    v4_m = v4_methods[v4_methods["mixer_method"] == mixer]
    v5_m = v5_methods[v5_methods["mixer_method"] == mixer]
    if v4_m.empty and v5_m.empty:
        continue
    if not v4_m.empty:
        v4_by_case = best_rows(v4_m, ["case"])
        mixer_rows.append({"version": "V4", "mixer_method": mixer, "mean_best_P_valid": float(v4_by_case["P_valid"].mean()), "median_best_P_valid": float(v4_by_case["P_valid"].median()), "cases": len(v4_by_case)})
        mixer_case_rows.append(add_version(v4_by_case, "V4"))
    if not v5_m.empty:
        v5_by_case = best_rows(v5_m, ["case"])
        mixer_rows.append({"version": "V5", "mixer_method": mixer, "mean_best_P_valid": float(v5_by_case["P_valid"].mean()), "median_best_P_valid": float(v5_by_case["P_valid"].median()), "cases": len(v5_by_case)})
        mixer_case_rows.append(add_version(v5_by_case, "V5"))

mixer_comparison = pd.DataFrame(mixer_rows)
mixer_detail = pd.concat(mixer_case_rows, ignore_index=True) if mixer_case_rows else pd.DataFrame()
mixer_comparison.to_csv(OUT_DIR / "v4_v5_mixer_comparison.csv", index=False)
mixer_detail.to_csv(OUT_DIR / "v4_v5_mixer_by_case_detail.csv", index=False)
print(mixer_comparison.to_string(index=False))

mixers = ordered_unique(mixer_comparison["mixer_method"], MIXER_ORDER)
x_mix = np.arange(len(mixers))
fig, ax = plt.subplots(figsize=(9, 5))
for offset, version, color in [(-0.18, "V4", V4_COLOR), (0.18, "V5", V5_COLOR)]:
    vals = []
    for mixer in mixers:
        row = mixer_comparison[(mixer_comparison["version"] == version) & (mixer_comparison["mixer_method"] == mixer)]
        vals.append(float(row["mean_best_P_valid"].iloc[0]) if not row.empty else np.nan)
    ax.bar(x_mix + offset, vals, width=0.34, color=color, label=version)
ax.set_xticks(x_mix)
ax.set_xticklabels(mixers, rotation=20, ha="right")
ax.set_ylabel("mean best P_valid across use cases")
ax.set_ylim(0, 1.05)
ax.set_title("Mixer method comparison")
ax.legend()
fig.tight_layout()
savefig(fig, "v4_v5_mixer_methods_mean")


version    mixer_method  mean_best_P_valid  median_best_P_valid  cases
     V4 local_geometric           0.906717             0.925754     10
     V5 local_geometric           0.922460             0.932688     10
     V4    linear_valid           0.868714             0.909922     10
     V5    linear_valid           0.910579             0.932688     10
     V4          rx_all           0.793805             0.810933     10
     V5          rx_all           0.827146             0.858800     10


In [8]:
# =========================================================
# V5 transition-mode comparison, full-config rankings, and parameter summaries
# =========================================================

mode_rows = []
for mode in ordered_unique(v5_methods["transition_mode"], V5_MODE_ORDER):
    mode_df = v5_methods[v5_methods["transition_mode"] == mode]
    if mode_df.empty:
        continue
    by_case = best_rows(mode_df, ["case"])
    mode_rows.append({
        "transition_mode": mode,
        "mean_best_P_valid": float(by_case["P_valid"].mean()),
        "median_best_P_valid": float(by_case["P_valid"].median()),
        "cases": len(by_case),
        "most_common_best_mixer": by_case["mixer_method"].mode().iloc[0] if not by_case["mixer_method"].mode().empty else "",
        "most_common_best_method": by_case["method"].mode().iloc[0] if not by_case["method"].mode().empty else "",
    })
mode_comparison = pd.DataFrame(mode_rows)
mode_comparison.to_csv(OUT_DIR / "v5_transition_mode_comparison.csv", index=False)
print(mode_comparison.to_string(index=False))

x_mode = np.arange(len(mode_comparison))
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x_mode, mode_comparison["mean_best_P_valid"], color=V5_ORIENTED_COLOR, alpha=0.9)
for xi, value in zip(x_mode, mode_comparison["mean_best_P_valid"]):
    ax.text(xi, min(1.04, value + 0.025), f"{value:.3f}", ha="center", va="bottom")
ax.set_xticks(x_mode)
ax.set_xticklabels(mode_comparison["transition_mode"], rotation=20, ha="right")
ax.set_ylabel("mean best P_valid across use cases")
ax.set_ylim(0, 1.05)
ax.set_title("V5 transition mode comparison")
fig.tight_layout()
savefig(fig, "v5_transition_modes_mean")

# Full-configuration ranking derived from param_methods_all_cases.
def config_summary(df, version):
    group_cols = ["transition_mode", "mixer_method", "method"]
    rows = []
    for keys, part in df.groupby(group_cols, dropna=False, observed=False):
        transition_mode, mixer_method, method = keys
        case_best = best_rows(part, ["case"])
        values = case_best["P_valid"].to_numpy(dtype=float)
        rows.append({
            "version": version,
            "transition_mode": transition_mode,
            "mixer_method": mixer_method,
            "method": method,
            "mean_best_P_valid": float(np.mean(values)),
            "median_best_P_valid": float(np.median(values)),
            "min_best_P_valid": float(np.min(values)),
            "max_best_P_valid": float(np.max(values)),
            "wins_best_by_case": int(sum(
                (case_comparison[f"{version}_best_transition_mode"] == transition_mode) &
                (case_comparison[f"{version}_best_mixer"] == mixer_method) &
                (case_comparison[f"{version}_best_method"] == method)
            )) if version in ["V4", "V5"] else 0,
            "cases": int(len(case_best)),
        })
    return pd.DataFrame(rows).sort_values(["mean_best_P_valid", "median_best_P_valid"], ascending=False).reset_index(drop=True)

v4_config_ranking = config_summary(v4_methods, "V4")
v5_config_ranking = config_summary(v5_methods, "V5")
full_config_ranking = (
    pd.concat([v4_config_ranking, v5_config_ranking], ignore_index=True)
    .sort_values(["mean_best_P_valid", "median_best_P_valid", "min_best_P_valid"], ascending=False)
    .reset_index(drop=True)
)
full_config_ranking.to_csv(OUT_DIR / "v4_v5_full_config_ranking.csv", index=False)
print("\nTop full configurations by mean best P_valid:")
print(full_config_ranking.head(20).to_string(index=False))

# Compare notebook-generated combo summary CSVs when present.
if v4_all_config_summary_file is not None:
    v4_all_config_summary_file.assign(version="V4").to_csv(OUT_DIR / "v4_all_config_summary_loaded.csv", index=False)
if v5_all_config_summary_file is not None:
    v5_all_config_summary_file.assign(version="V5").to_csv(OUT_DIR / "v5_all_config_summary_loaded.csv", index=False)

fig, ax = plt.subplots(figsize=(12, 6))
top = full_config_ranking.head(12).copy()
labels = [f"{row.version}\n{row.transition_mode}\n{row.mixer_method}\n{row.method}" for row in top.itertuples()]
colors = [V4_COLOR if version == "V4" else V5_COLOR for version in top["version"]]
y = np.arange(len(top))
ax.barh(y, top["mean_best_P_valid"], color=colors, alpha=0.9)
ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("mean best P_valid across use cases")
ax.set_xlim(0, 1.05)
ax.set_title("Top full configurations across V4 and V5")
fig.tight_layout()
savefig(fig, "v4_v5_top_full_configurations")

parameter_summary = pd.concat([
    add_version(v4_best_case, "V4"),
    add_version(v5_best_case, "V5"),
], ignore_index=True)
keep_cols = ["version", "case", "P_valid", "method", "mixer_method", "transition_mode", "theta_over_pi", "beta_over_pi", "r", "P_uniform", "W", "valid_windows"]
parameter_summary = parameter_summary[[col for col in keep_cols if col in parameter_summary.columns]]
parameter_summary.to_csv(OUT_DIR / "v4_v5_best_parameter_summary.csv", index=False)
print("\nBest parameters by version/use case:")
print(parameter_summary.to_string(index=False))


transition_mode  mean_best_P_valid  median_best_P_valid  cases most_common_best_mixer most_common_best_method
     delta_cost           0.858713             0.872087     10           linear_valid             M5 (hybrid)
        descent           0.890108             0.932520     10        local_geometric             M5 (hybrid)
 valid_boundary           0.840972             0.868241     10           linear_valid             M5 (hybrid)
 oriented_valid           0.911581             0.925703     10        local_geometric             M5 (hybrid)

Top full configurations by mean best P_valid:
version transition_mode    mixer_method       method  mean_best_P_valid  median_best_P_valid  min_best_P_valid  max_best_P_valid  wins_best_by_case  cases
     V4      cost_phase local_geometric  M5 (hybrid)           0.906717             0.925754          0.760868          0.986495                  6     10
     V5  oriented_valid local_geometric  M5 (hybrid)           0.905795             0.902883 

In [9]:
# =========================================================
# Compact conclusion
# =========================================================

summary = {
    "full_sweep_V4_mean_best_P_valid": float(case_comparison["V4_best_P_valid"].mean()),
    "full_sweep_V5_mean_best_P_valid": float(case_comparison["V5_best_P_valid"].mean()),
    "full_sweep_mean_delta_V5_minus_V4": float(case_comparison["delta_V5_minus_V4"].mean()),
    "full_sweep_V5_beats_V4_cases": int((case_comparison["delta_V5_minus_V4"] > 0).sum()),
    "default_mixer_V4_mean_P_valid": float(default_mixer_case_comparison["V4_P_opt_heatmap"].mean()),
    "default_mixer_V5_mean_P_valid": float(default_mixer_case_comparison["V5_P_opt_heatmap"].mean()),
    "default_mixer_mean_delta_V5_minus_V4": float(default_mixer_case_comparison["delta_V5_minus_V4"].mean()),
    "default_mixer_V5_beats_V4_cases": int((default_mixer_case_comparison["delta_V5_minus_V4"] > 0).sum()),
    "total_cases": int(len(case_comparison)),
    "full_sweep_best_V4_mixer_counts": case_comparison["V4_best_mixer"].value_counts().to_dict(),
    "full_sweep_best_V5_mixer_counts": case_comparison["V5_best_mixer"].value_counts().to_dict(),
    "full_sweep_best_V5_transition_mode_counts": case_comparison["V5_best_transition_mode"].value_counts().to_dict(),
    "default_mixer_best_V5_transition_mode_counts": default_mixer_case_comparison["V5_best_transition_mode"].value_counts().to_dict(),
}
print(summary)

labels = ["V4 full", "V5 full", "V4 default", "V5 default"]
values = [
    summary["full_sweep_V4_mean_best_P_valid"],
    summary["full_sweep_V5_mean_best_P_valid"],
    summary["default_mixer_V4_mean_P_valid"],
    summary["default_mixer_V5_mean_P_valid"],
]
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(labels, values, color=[V4_COLOR, V5_COLOR, V4_COLOR, V5_COLOR], alpha=0.9)
for bar, value in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 0.02, f"{value:.3f}", ha="center", va="bottom", fontsize=11)
ax.set_ylabel("mean P_valid")
ax.set_ylim(0, 1.05)
ax.set_title("Overall V4 vs V5 performance: full sweep vs default-mixer reference")
fig.tight_layout()
savefig(fig, "v4_v5_overall_mean")

pd.DataFrame([summary]).to_csv(OUT_DIR / "v4_v5_overall_summary.csv", index=False)


{'full_sweep_V4_mean_best_P_valid': 0.9070347691818663, 'full_sweep_V5_mean_best_P_valid': 0.922532566841532, 'full_sweep_mean_delta_V5_minus_V4': 0.015497797659665635, 'full_sweep_V5_beats_V4_cases': 6, 'default_mixer_V4_mean_P_valid': 0.826332997626181, 'default_mixer_V5_mean_P_valid': 0.8429394752549928, 'default_mixer_mean_delta_V5_minus_V4': 0.016606477628811688, 'default_mixer_V5_beats_V4_cases': 6, 'total_cases': 10, 'full_sweep_best_V4_mixer_counts': {'local_geometric': 7, 'linear_valid': 2, 'rx_all': 1}, 'full_sweep_best_V5_mixer_counts': {'local_geometric': 8, 'linear_valid': 1, 'rx_all': 1}, 'full_sweep_best_V5_transition_mode_counts': {'oriented_valid': 7, 'descent': 3}, 'default_mixer_best_V5_transition_mode_counts': {'oriented_valid': 6, 'descent': 3, 'delta_cost': 1}}
